# Networks: check your answers in Python (Dijkstra, Prim, Kruskal)

Lecture: *Week 8 - Networks - Shortest Paths and MST*. Tutorial: *Week 8 - Tutorial - Networks - Shortest Paths and MST*,
Q1 (minimum spanning tree) and Q2 (shortest paths).

**The problem.** This week is done with pen and paper. This notebook runs the same three algorithms as the slides,
step by step, so you can check your tables:

- **Dijkstra** (shortest path from one vertex): prints, at each step, the vertex that becomes *permanent* and the
  labels $d_v$ (distance) and $\pi_v$ (predecessor), as in the slide tables.
- **Prim** and **Kruskal** (minimum spanning tree): print the order in which vertices or edges are selected and
  the total weight.

**How to run it.** Plain Python, nothing to install: choose *Run → Run All Cells*. Try the tutorial by hand first.

**Expected results.** Lecture: shortest path s–d–b–e–t of length **15**; MST weight **21** (Prim and Kruskal).
Tutorial Q1: MST weight **39**. Q2a: 1–3–4, length **5**. Q2b: 1–3–4–2, length **55**. The last section runs
Dijkstra on the *negative weight example* and shows it giving the wrong answer.

## 1. Dijkstra's algorithm

A directed network is stored as a dictionary of edges: `(i, j): w` means an edge from `i` to `j` with weight `w`.
The function follows the four steps on the slides: start with $d_s = 0$ and every other $d_v = \infty$ (all
temporary); make the temporary vertex with the smallest $d_v$ permanent; update the labels of the temporary
vertices it reaches; repeat until every vertex is permanent. Ties are broken by the order in which the vertices
are listed, so your choice in a tie may differ from the printout while the distances stay the same.

In [1]:
INF = float("inf")

def dijkstra(vertices, edges, source, show=True):
    d = {v: INF for v in vertices}     # distance labels d_v
    pred = {v: "-" for v in vertices}  # predecessor labels pi_v
    d[source] = 0
    temporary = list(vertices)         # all vertices start temporary
    while temporary:
        # step 2: the temporary vertex with the smallest distance becomes permanent
        u = min(temporary, key=lambda v: d[v])
        temporary.remove(u)
        changed = []
        # step 3: update temporary vertices reachable from u along an edge (u, v)
        for (i, j), w in edges.items():
            if i == u and j in temporary and d[u] + w < d[j]:
                d[j] = d[u] + w
                pred[j] = u
                changed.append(f"{j}: {d[j]} via {u}")
        if show:
            print(f"permanent {u} (d = {d[u]}); updated: {', '.join(changed) or 'none'}")
    return d, pred

def path_to(pred, target):
    # work backwards from the destination along the predecessors
    path = [target]
    while pred[path[-1]] != "-":
        path.append(pred[path[-1]])
    return path[::-1]

def print_table(vertices, d, pred):
    print("v   d_v  pi_v")
    for v in vertices:
        print(f"{v!s:3} {d[v]!s:>3}  {pred[v]}")

**Lecture example** (slides 8–28): vertices s, b, c, d, e, t. Note the two edges between d and e, one in each
direction (8 and 1).

In [2]:
V = ["s", "b", "c", "d", "e", "t"]
E = {("s", "b"): 9, ("s", "d"): 2,
     ("b", "c"): 11, ("b", "e"): 3,
     ("d", "b"): 4, ("d", "e"): 8,
     ("e", "d"): 1, ("e", "t"): 6,
     ("c", "e"): 5, ("c", "t"): 12}
d, pred = dijkstra(V, E, "s")
print()
print_table(V, d, pred)
print("\nshortest path s -> t:", "-".join(path_to(pred, "t")), "length", d["t"])

permanent s (d = 0); updated: b: 9 via s, d: 2 via s
permanent d (d = 2); updated: b: 6 via d, e: 10 via d
permanent b (d = 6); updated: c: 17 via b, e: 9 via b
permanent e (d = 9); updated: t: 15 via e
permanent t (d = 15); updated: none
permanent c (d = 17); updated: none

v   d_v  pi_v
s     0  -
b     6  d
c    17  b
d     2  s
e     9  b
t    15  e

shortest path s -> t: s-d-b-e-t length 15


**Tutorial Q2a** (node 1 to node 4) and **Q2b** (node 1 to node 2).

In [3]:
V2a = [1, 2, 3, 4]
E2a = {(1, 2): 2, (1, 3): 3, (2, 3): 1, (2, 4): 4, (3, 4): 2}
d, pred = dijkstra(V2a, E2a, 1)
print("Q2a: path", "-".join(map(str, path_to(pred, 4))), "length", d[4])
print()

V2b = [1, 2, 3, 4, 5]
E2b = {(1, 2): 100, (1, 3): 30, (2, 3): 20, (3, 4): 10,
       (3, 5): 60, (4, 2): 15, (4, 5): 50}
d, pred = dijkstra(V2b, E2b, 1)
print("Q2b: path", "-".join(map(str, path_to(pred, 2))), "length", d[2])

permanent 1 (d = 0); updated: 2: 2 via 1, 3: 3 via 1
permanent 2 (d = 2); updated: 4: 6 via 2
permanent 3 (d = 3); updated: 4: 5 via 3
permanent 4 (d = 5); updated: none
Q2a: path 1-3-4 length 5

permanent 1 (d = 0); updated: 2: 100 via 1, 3: 30 via 1
permanent 3 (d = 30); updated: 4: 40 via 3, 5: 90 via 3
permanent 4 (d = 40); updated: 2: 55 via 4
permanent 2 (d = 55); updated: none
permanent 5 (d = 90); updated: none
Q2b: path 1-3-4-2 length 55


## 2. Prim's algorithm

An undirected network is a dictionary of edges `(u, v): w`; each edge can be used in both directions. Prim starts
from one vertex and keeps adding the cheapest edge from a selected vertex to an unselected one. The `start` vertex
is arbitrary: the slides start from D, and any start gives the same total weight.

In [4]:
def prim(vertices, edges, start):
    selected = [start]
    tree = []
    while len(selected) < len(vertices):
        # every edge with exactly one end selected could join the tree
        candidates = [(w, u, v) for (u, v), w in edges.items()
                      if (u in selected) != (v in selected)]
        w, u, v = min(candidates)   # the cheapest one (ties: the first in sorted order)
        new = v if u in selected else u
        selected.append(new)
        tree.append((u, v, w))
        print(f"select {new} via edge ({u},{v}) of weight {w}")
    total = sum(w for _, _, w in tree)
    print("total weight", total)
    return tree, total

**Lecture example** (slides 39–55), started at D as on the slides.

In [5]:
VL = list("ABCDEFGH")
EL = {("A", "F"): 10, ("A", "B"): 8, ("A", "H"): 4, ("F", "C"): 3, ("F", "B"): 7,
      ("F", "D"): 18, ("F", "E"): 2, ("C", "D"): 3, ("C", "B"): 4, ("B", "E"): 10,
      ("B", "H"): 9, ("G", "D"): 2, ("D", "E"): 25, ("H", "G"): 3, ("G", "E"): 7}
tree, total = prim(VL, EL, "D")

select G via edge (G,D) of weight 2
select C via edge (C,D) of weight 3
select F via edge (F,C) of weight 3
select E via edge (F,E) of weight 2
select H via edge (H,G) of weight 3
select A via edge (A,H) of weight 4
select B via edge (C,B) of weight 4
total weight 21


## 3. Kruskal's algorithm

Kruskal sorts the edges from lightest to heaviest and selects an edge unless it would close a cycle. To spot a
cycle it remembers which *component* (group of connected vertices) each vertex is in: an edge whose two ends are
already in the same component would close a cycle. It stops after $|V| - 1$ edges.

In [6]:
def kruskal(vertices, edges):
    component = {v: v for v in vertices}   # each vertex starts in its own component

    def find(v):
        # follow the links to the vertex that names v's component
        while component[v] != v:
            v = component[v]
        return v

    tree = []
    for (u, v), w in sorted(edges.items(), key=lambda e: e[1]):  # lightest first
        cu, cv = find(u), find(v)
        if cu == cv:
            print(f"({u},{v}) {w}: F (forms a cycle)")
        else:
            component[cu] = cv               # join the two components
            tree.append((u, v, w))
            print(f"({u},{v}) {w}: T")
            if len(tree) == len(vertices) - 1:
                break                         # |V| - 1 edges: stop
    total = sum(w for _, _, w in tree)
    print("total weight", total)
    return tree, total

**Lecture example** (slides 56–69). The table on the slides is used, with (B,E) = 4. Edges are listed in the
slides' order so that ties are taken in the same order.

In [7]:
EK = {("D", "E"): 1, ("D", "G"): 2, ("E", "G"): 3, ("C", "D"): 3, ("G", "H"): 3,
      ("C", "F"): 3, ("B", "C"): 4, ("B", "E"): 4, ("B", "F"): 4, ("B", "H"): 4,
      ("A", "H"): 5, ("D", "F"): 6, ("A", "B"): 8, ("A", "F"): 10}
tree, total = kruskal(VL, EK)

(D,E) 1: T
(D,G) 2: T
(E,G) 3: F (forms a cycle)
(C,D) 3: T
(G,H) 3: T
(C,F) 3: T
(B,C) 4: T
(B,E) 4: F (forms a cycle)
(B,F) 4: F (forms a cycle)
(B,H) 4: F (forms a cycle)
(A,H) 5: T
total weight 21


**Tutorial Q1** with both algorithms (Prim started at A, as in the answers).

In [8]:
V1 = list("ABCDEFG")
E1 = {("A", "B"): 7, ("A", "D"): 5, ("B", "C"): 8, ("B", "E"): 7, ("B", "D"): 9,
      ("C", "E"): 5, ("D", "E"): 15, ("D", "F"): 6, ("E", "F"): 8, ("E", "G"): 9,
      ("F", "G"): 11}
print("Prim:")
tree_p, total_p = prim(V1, E1, "A")
print("\nKruskal:")
tree_k, total_k = kruskal(V1, E1)

Prim:
select D via edge (A,D) of weight 5
select F via edge (D,F) of weight 6
select B via edge (A,B) of weight 7
select E via edge (B,E) of weight 7
select C via edge (C,E) of weight 5
select G via edge (E,G) of weight 9
total weight 39

Kruskal:
(A,D) 5: T
(C,E) 5: T
(D,F) 6: T
(A,B) 7: T
(B,E) 7: T
(B,C) 8: F (forms a cycle)
(E,F) 8: F (forms a cycle)
(B,D) 9: F (forms a cycle)
(E,G) 9: T
total weight 39


## 4. When Dijkstra fails: a negative edge weight

The *Negative weight example* deck uses this network, with one edge of weight −5. Dijkstra makes vertex 2
permanent with distance 1 before it has looked at the edge 3 → 4, and never changes a permanent label again.
The true shortest path 1–3–4–2 has length 3 + (−5) + 1 = −1. (The deck also shows vertex 4's label changing to
−2 after it was made permanent; the code below, like the algorithm, does not change permanent labels.)

In [9]:
Vn = [1, 2, 3, 4]
En = {(1, 2): 1, (1, 4): 2, (1, 3): 3, (3, 4): -5, (4, 2): 1}
d, pred = dijkstra(Vn, En, 1)
print("Dijkstra says 1 -> 2:", "-".join(map(str, path_to(pred, 2))), "length", d[2])
print("but 1-3-4-2 has length", En[(1, 3)] + En[(3, 4)] + En[(4, 2)])

permanent 1 (d = 0); updated: 2: 1 via 1, 4: 2 via 1, 3: 3 via 1
permanent 2 (d = 1); updated: none
permanent 4 (d = 2); updated: none
permanent 3 (d = 3); updated: none
Dijkstra says 1 -> 2: 1-2 length 1
but 1-3-4-2 has length -1


## What do you notice?

In Q2a, run `dijkstra(V2a, E2a, 1)` again and look at the step where vertex 2 is made permanent. Is 1–3–4 the
only shortest path from 1 to 4?

<details><summary>Answer</summary>

No. When vertex 2 becomes permanent, 2 + 1 = 3 equals the label 3 that vertex 3 already has, so the label is not
changed (only a *strictly* shorter route changes it). The path 1–2–3–4 also has length 2 + 1 + 2 = **5**. Both
answers are right; the answer sheet lists 1–3–4. In the same way, in Q2b vertex 5 can be reached by 1–3–5 or
1–3–4–5, both of length 90.
</details>

## Try it

Change the weight of the edge (3, 4) in the negative example from −5 to 5 and run the last code cell again.
Does Dijkstra now give the right answer?

<details><summary>Answer</summary>

Yes. With every weight nonnegative Dijkstra is correct: it returns 1–2 with length 1, which is the shortest path
(1–3–4–2 now has length 9).
</details>